# NLRB union elections — exploratory data analysis

Loads the `analytical_elections` view (RC/RM filings) and profiles the two
modeling targets: the regression target `vote_ratio` and the classification
target `success`.

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

con = duckdb.connect("../data/union_elections.duckdb", read_only=True)
df = con.execute("SELECT * FROM analytical_elections").fetchdf()
print(df.shape)
df.head()

## Grain and coverage

The view can contain several `election_mode` records per election, so it is
not strictly one row per election.

In [ ]:
print("rows:", len(df))
print("distinct filings:", df["case_number"].nunique())
print("distinct elections:", df["election_id"].nunique())
print()
df["case_type"].value_counts()

## Closing reason and the success label

An election is treated as successful when
`reason_closed == 'Certific. of Representative'`.

In [ ]:
df["reason_closed"].value_counts(dropna=False)

In [ ]:
df["success"] = (
    df["reason_closed"]
    .map(lambda value: pd.NA if pd.isna(value) else int(value == "Certific. of Representative"))
    .astype("Int64")
)
df["success"].value_counts(dropna=False)

## Vote counts and the regression target

`vote_ratio = votes_for_labor_organization_1 / votes_cast_against_labor_org`.
Elections with zero votes against give an undefined ratio.

In [ ]:
votes = df.dropna(
    subset=["votes_for_labor_organization_1", "votes_cast_against_labor_org"]
).copy()
print("elections with vote counts:", len(votes))
print("zero votes against:", int((votes["votes_cast_against_labor_org"] == 0).sum()))

votes["vote_ratio"] = (
    votes["votes_for_labor_organization_1"]
    / votes["votes_cast_against_labor_org"].replace(0, float("nan"))
)
votes["vote_ratio"].describe()

## Missingness

In [ ]:
df.isna().mean().sort_values(ascending=False).head(30)

## Finance coverage

In [ ]:
(
    df.groupby("form_type", dropna=False)
      .agg(
          rows=("case_number", "size"),
          median_disbursements=("ttl_disbursements", "median"),
          median_receipts=("ttl_receipts", "median"),
      )
      .sort_values("rows", ascending=False)
)

## Success by case type and region

In [ ]:
df.groupby("case_type")["success"].agg(["size", "mean"])

In [ ]:
(
    df.groupby("region_assigned")["success"]
      .agg(["size", "mean"])
      .sort_values("size", ascending=False)
      .head(15)
)

## Notes

- Vote counts exist only for elections with an `election_mode` record, so the
  regression sample is much smaller than the classification sample.
- About a quarter of elections with vote counts have zero votes against, so
  their ratio is undefined and they are dropped for regression.
- `training.py` loads the same view, collapses it to one row per election, and
  builds the feature/target frames.